# 01 - Model Inference on IEMOCAP

Goal: load IEMOCAP audio samples and run the pretrained SpeechBrain wav2vec2-IEMOCAP emotion classifier.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import pandas as pd

from speech_xai_project import audio, config, model, plotting

project_config = config.load_config(PROJECT_ROOT / 'configs' / 'default.yaml')
IEMOCAP_ROOT = config.project_path(project_config['paths']['iemocap_root'])
METADATA_CSV = config.project_path(project_config['paths']['metadata_csv'])
PREDICTIONS_CSV = config.project_path(project_config['paths']['predictions_csv'])
MODEL_SOURCE = project_config['model']['speechbrain_source']
MODEL_SAVEDIR = config.project_path(project_config['model']['savedir'])
TARGET_SAMPLE_RATE = project_config['model']['sample_rate']

PREDICTIONS_CSV.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
if METADATA_CSV.exists():
    metadata_table = pd.read_csv(METADATA_CSV)
else:
    wav_paths = sorted(IEMOCAP_ROOT.rglob('*.wav'))[:20]
    metadata_table = pd.DataFrame({
        'audio_id': [path.stem for path in wav_paths],
        'audio_path': [str(path) for path in wav_paths],
        'true_label': [None for _ in wav_paths],
    })

metadata_table.head()

In [ ]:
sample_row = metadata_table.iloc[0]
sample_waveform, sample_rate = audio.load_waveform(sample_row.audio_path, TARGET_SAMPLE_RATE)
print('audio_id:', sample_row.audio_id)
print('duration:', audio.duration_seconds(sample_waveform, sample_rate))
plotting.plot_waveform(sample_waveform, sample_rate, title=str(sample_row.audio_id));

In [ ]:
classifier = model.load_speechbrain_classifier(MODEL_SOURCE, MODEL_SAVEDIR)
prediction_rows = []

for row in metadata_table.head(5).itertuples(index=False):
    waveform, sample_rate = audio.load_waveform(row.audio_path, TARGET_SAMPLE_RATE)
    prediction = model.classify_waveform(classifier, audio.waveform_to_batch(waveform))
    prediction_rows.append({
        'audio_id': row.audio_id,
        'audio_path': row.audio_path,
        'true_label': getattr(row, 'true_label', None),
        'predicted_label': prediction.predicted_label,
        'predicted_index': prediction.predicted_index,
        'predicted_confidence': prediction.predicted_confidence,
    })

predictions_table = pd.DataFrame(prediction_rows)
predictions_table.to_csv(PREDICTIONS_CSV, index=False)
predictions_table

## Current status

- Tested: SpeechBrain classifier inference on a few audio files.
- Works: fill in after running.
- Failed: fill in after running.
- Generated: `results/predictions.csv`.
- Next: inspect attention access in Notebook 02.